In [1]:
import pandas as pd
import numpy as np
import re
from pathlib import Path

In [2]:
file_path = Path(
    r"C:\Users\praso\Downloads\Saudi Employee Attrition Dataset\01_Raw_Data\Original Dataset of Employee Attrition.xlsx"
)

df = pd.read_excel(file_path)
df_clean = df.copy()

print("Shape:", df_clean.shape)

Shape: (1191, 35)


In [3]:
def clean_column_name(col):
    col = str(col).replace("\xa0", " ").strip()
    col = re.sub(r"[^\w]+", "_", col)
    return col.strip("_")

df_clean.columns = [clean_column_name(col) for col in df_clean.columns]

print(df_clean.columns.tolist())

['ID', 'Attrition', 'Gender', 'Age', 'Maritalstatus', 'Academic_degree', 'Years_Experience', 'Years_experience_lastorganization', 'Sector', 'Department', 'JobTitle', 'MonthlySalary', 'Allowances', 'MedicalInsurance', 'Bonus', 'OverTime', 'Payment_Overtime', 'Rewards_Wages_Satisfaction', 'Get__Deserved_Promotion', 'Training_programs__During_last_three_years', 'Useful_Training_Programs', 'Business_Travel', 'Job_Support', 'Recognition', 'Emotional_Commitment', 'Job_Engagement', 'Distance_to_work', 'Work_Live_Balance', 'Physical_Stress', 'Psychological_Exhaustion', 'Job_Stability', 'Health_Issues', 'Environment_Satisfaction', 'Job_Satisfaction', 'Job_Opportunities']


In [4]:
text_columns = df_clean.select_dtypes(include="object").columns

for col in text_columns:
    df_clean[col] = (
        df_clean[col]
        .astype("string")
        .str.strip()
        .str.replace(r"\s+", " ", regex=True)
    )

In [5]:
df_clean["Sector"] = df_clean["Sector"].replace({
    "education sector": "Education sector"
})

df_clean["Department"] = df_clean["Department"].replace({
    "patient Affairs": "Patient Affairs",
    "technical support": "Technical support"
})

df_clean["JobTitle"] = df_clean["JobTitle"].replace({
    "it specialist": "IT Specialist",
    "nurse": "Nurse"
})

In [6]:
df_clean["AttritionFlag"] = df_clean["Attrition"].map({
    "No": 0,
    "Yes": 1
})

print(df_clean["AttritionFlag"].value_counts())

AttritionFlag
0    676
1    515
Name: count, dtype: int64


In [7]:
mappings = {
    "Job_Satisfaction": {
        "Not satisfied": 0,
        "Satisfied": 1,
        "Very satisfied": 2
    },
    "Environment_Satisfaction": {
        "Low": 0,
        "Medium": 1,
        "High": 2
    },
    "Emotional_Commitment": {
        "Low": 0,
        "Medium": 1,
        "High": 2
    },
    "Job_Support": {
        "Low": 0,
        "Medium": 1,
        "High": 2
    },
    "Physical_Stress": {
        "No": 0,
        "Sometime": 1,
        "Yes": 2
    },
    "Psychological_Exhaustion": {
        "No": 0,
        "Sometime": 1,
        "Yes": 2
    }
}

for col, mapping in mappings.items():
    df_clean[f"{col}_Score"] = df_clean[col].map(mapping)

In [8]:
df_clean["AgeOrder"] = df_clean["Age"].map({
    "21 to 30": 0,
    "31 to 40": 1,
    "41 to 50": 2,
    "51 to 60": 3
})

df_clean["ExperienceOrder"] = df_clean["Years_Experience"].map({
    "Less than 5 years": 0,
    "From 5 to 10 years": 1,
    "From 11 to 15 years": 2,
    "From 16 to 20 years": 3,
    "From 21 to 25 years": 4,
    "From 26 to 30 years": 5,
    "From 31 to 35 years": 6
})

In [9]:
print("Rows:", len(df_clean))
print("Columns:", len(df_clean.columns))
print("Missing values:", df_clean.isna().sum().sum())
print("Duplicate rows:", df_clean.duplicated().sum())
print("Duplicate IDs:", df_clean["ID"].duplicated().sum())

Rows: 1191
Columns: 44
Missing values: 880
Duplicate rows: 0
Duplicate IDs: 0


In [10]:
for col in ["Attrition", "Gender", "Sector", "Department", "Job_Satisfaction"]:
    print(f"\n{col}:")
    print(df_clean[col].value_counts(dropna=False))


Attrition:
Attrition
No     676
Yes    515
Name: count, dtype: Int64

Gender:
Gender
Female    678
Male      513
Name: count, dtype: Int64

Sector:
Sector
Medical sector                                304
Education sector                              272
Financial sector                              153
Food production sector                        114
Transport sector                               69
Communications and IT sector                   64
Economic sector                                48
Energy sector                                  47
Environment, Water, and Agriculture sector     27
Industry sector                                27
Tourism sector                                 26
Engineering consulting companies               14
Media sector                                   13
Law firm                                       11
Restaurant sector                               2
Name: count, dtype: Int64

Department:
Department
Administration            194
Teaching      

In [11]:
from pathlib import Path

output_path = Path(
    r"C:\Users\praso\Downloads\Saudi Employee Attrition Dataset\02_Cleaned_Data\employee_attrition_clean.csv"
)

df_clean.to_csv(
    output_path,
    index=False,
    encoding="utf-8-sig"
)

print("Saved:", output_path)
print("Final Shape:", df_clean.shape)

Saved: C:\Users\praso\Downloads\Saudi Employee Attrition Dataset\02_Cleaned_Data\employee_attrition_clean.csv
Final Shape: (1191, 44)


In [12]:
score_columns = [
    "Job_Satisfaction_Score",
    "Environment_Satisfaction_Score",
    "Emotional_Commitment_Score",
    "Job_Support_Score",
    "Physical_Stress_Score",
    "Psychological_Exhaustion_Score"
]

for col in score_columns:
    print(f"\n{col}")
    print(df_clean[col].isna().sum(), "missing")


Job_Satisfaction_Score
0 missing

Environment_Satisfaction_Score
0 missing

Emotional_Commitment_Score
0 missing

Job_Support_Score
0 missing

Physical_Stress_Score
450 missing

Psychological_Exhaustion_Score
430 missing


In [13]:
for col in [
    "Job_Satisfaction",
    "Environment_Satisfaction",
    "Emotional_Commitment",
    "Job_Support",
    "Physical_Stress",
    "Psychological_Exhaustion"
]:
    print(f"\n{col}")
    print(df_clean[col].unique())


Job_Satisfaction
<StringArray>
['Not satisfied', 'Satisfied', 'Very satisfied']
Length: 3, dtype: string

Environment_Satisfaction
<StringArray>
['Medium', 'Low', 'High']
Length: 3, dtype: string

Emotional_Commitment
<StringArray>
['Medium', 'Low', 'High']
Length: 3, dtype: string

Job_Support
<StringArray>
['Medium', 'Low', 'High']
Length: 3, dtype: string

Physical_Stress
<StringArray>
['Yes', 'No', 'Sometimes']
Length: 3, dtype: string

Psychological_Exhaustion
<StringArray>
['No', 'Sometimes', 'Yes']
Length: 3, dtype: string


In [14]:
df_clean["Physical_Stress_Score"] = df_clean["Physical_Stress"].map({
    "No": 0,
    "Sometimes": 1,
    "Yes": 2
})

df_clean["Psychological_Exhaustion_Score"] = df_clean["Psychological_Exhaustion"].map({
    "No": 0,
    "Sometimes": 1,
    "Yes": 2
})

In [15]:
print("Physical Stress missing:",
      df_clean["Physical_Stress_Score"].isna().sum())

print("Psychological Exhaustion missing:",
      df_clean["Psychological_Exhaustion_Score"].isna().sum())

Physical Stress missing: 0
Psychological Exhaustion missing: 0


In [16]:
score_columns = [
    "Job_Satisfaction_Score",
    "Environment_Satisfaction_Score",
    "Emotional_Commitment_Score",
    "Job_Support_Score",
    "Physical_Stress_Score",
    "Psychological_Exhaustion_Score"
]

print(df_clean[score_columns].isna().sum())

Job_Satisfaction_Score            0
Environment_Satisfaction_Score    0
Emotional_Commitment_Score        0
Job_Support_Score                 0
Physical_Stress_Score             0
Psychological_Exhaustion_Score    0
dtype: int64


In [22]:
output_path = Path(
    r"C:\Users\praso\Downloads\Saudi Employee Attrition Dataset\02_Cleaned_Data\employee_attrition_clean.csv"
)

df_clean.to_csv(
    output_path,
    index=False,
    encoding="utf-8-sig"
)

print("Saved:", output_path)
print("Final Shape:", df_clean.shape)

Saved: C:\Users\praso\Downloads\Saudi Employee Attrition Dataset\02_Cleaned_Data\employee_attrition_clean.csv
Final Shape: (1191, 44)


In [23]:
print("Shape:", df_clean.shape)

print("\nMissing values:")
print(df_clean.isna().sum().sum())

print("\nDuplicate rows:")
print(df_clean.duplicated().sum())

print("\nDuplicate IDs:")
print(df_clean["ID"].duplicated().sum())

print("\nAttrition:")
print(df_clean["Attrition"].value_counts())

print("\nAttrition Flag:")
print(df_clean["AttritionFlag"].value_counts())

Shape: (1191, 44)

Missing values:
0

Duplicate rows:
0

Duplicate IDs:
0

Attrition:
Attrition
No     676
Yes    515
Name: count, dtype: Int64

Attrition Flag:
AttritionFlag
0    676
1    515
Name: count, dtype: int64


In [20]:
df_clean["JobTitle"] = (
    df_clean["JobTitle"]
    .str.strip()
    .str.lower()
    .replace({
        "it specialist": "IT Specialist",
        "nurse": "Nurse",
        "salesman": "Salesman"
    })
)

In [21]:
print(
    sorted(
        [x for x in df_clean["JobTitle"].dropna().unique()
         if "it specialist" in x.lower()]
    )
)

print(
    sorted(
        [x for x in df_clean["JobTitle"].dropna().unique()
         if "salesman" in x.lower()]
    )
)

['IT Specialist']
['Salesman']
